In [0]:
%sql
USE CATALOG gold_dev;
USE SCHEMA analytics;


In [0]:
%sql
CREATE OR REPLACE TABLE sales_base AS
SELECT
    f.order_id,

    -- Dates
    od.date AS order_date,
    sd.date AS ship_date,
    DATEDIFF(sd.date, od.date) AS days_to_ship,

    -- Customer
    c.customer_key,
    c.customer_id,
    c.customer_name,
    c.customer_segment,

    -- Region from dim_region
    r.region AS region,

    -- Product
    p.product_key,
    p.product_id,
    p.product_name,
    p.category,
    p.sub_category,

    -- Measures
    f.order_quantity,
    f.sales_amount,
    f.discount_amount,
    f.profit_amount,

    -- Metadata
    f.ingestion_ts,
    f.load_timestamp

FROM silver_dev.global_mart_retail.fact_sales f

-- Current customer
JOIN silver_dev.global_mart_retail.dim_customer c
  ON f.customer_key = c.customer_key
 AND c.is_current_record = true

-- Current product
JOIN silver_dev.global_mart_retail.dim_product p
  ON f.product_key = p.product_key
 AND p.is_current_record = true

-- Order and Ship dates
JOIN silver_dev.global_mart_retail.dim_date od
  ON f.order_date_key = od.date_key

JOIN silver_dev.global_mart_retail.dim_date sd
  ON f.ship_date_key = sd.date_key

-- Region lookup
LEFT JOIN silver_dev.global_mart_retail.dim_region r
  ON f.region_key = r.region_key;
